# LABORATORIO N° 04

## Analítica diagnóstica: ¿por qué pasó?

Tema: Descomposición de variaciones, cohortes y segmentación explicativa
Laboratorio: Diagnóstico de una variación de ventas en un comercio electrónico británico, con datos reales

Analítica Empresarial Integrada · Pilar Rocío Sayán Mejía · Semestre 2026-II
---

**Equipo N.°:** _____ &nbsp;&nbsp;&nbsp; **Sección:** 6C28 &nbsp;&nbsp;&nbsp; **Fecha:** 09/09/26

**Apellidos y nombres del estudiante:** _(Salazar Chavez, Renzo Sebastian)_

**Integrantes del equipo:** _()_

---

---

## I. Elemento de la capacidad terminal de la semana

Explica la variación de un resultado empresarial mediante la descomposición de los factores que la componen. Distingue los niveles descriptivo, diagnóstico y causal del análisis, y evita atribuir causalidad a una asociación observada. Construye cohortes de clientes, calcula tasas de retención comparables entre grupos de igual antigüedad e identifica el driver dominante de una variación, formulando una recomendación empresarial sustentada en cifras verificables y en el reconocimiento explícito de sus limitaciones.

---

## II. Seguridad

- Abstenerse de manipular hardware, conexiones eléctricas o de red sin la supervisión correspondiente.
- No ingerir alimentos ni bebidas dentro del laboratorio.
- Mantener la estación de trabajo, el mobiliario y los equipos en condiciones de orden y limpieza.
- No descargar ni instalar software no autorizado.
- Ejecutar el código exclusivamente en el entorno Google Colab.
- No alterar ni fabricar valores de los datos; toda decisión metodológica debe quedar debidamente documentada.

---

## III. Fundamento teórico

Con carácter previo a la sesión de laboratorio, el estudiante deberá revisar el material de la Semana 4, así como los conceptos de analítica diagnóstica, árbol de drivers, descomposición precio-volumen-mezcla, análisis de cohortes, tasa de retención y segmentación explicativa.

- Provost, F. y Fawcett, T. (2013). Data Science for Business. Capítulos 2 y 3.
- Sharda, R., Delen, D. y Turban, E. (2024). Business Intelligence, Analytics, Data Science, and AI. Capítulo 3.
- Fader, P. S. y Hardie, B. G. S. (2009). Probability models for customer-base analysis. Journal of Interactive Marketing, 23(1), 61-69.
- Knaflic, C. N. (2015). Storytelling with Data. Capítulos 2 y 3.

---

## IV. Normas empleadas

No resulta aplicable una norma técnica específica. Se adoptan buenas prácticas de reproducibilidad y trazabilidad de las fuentes. En particular, el laboratorio verifica la integridad del conjunto de datos mediante su huella criptográfica SHA-256 antes de utilizarlo, y atribuye la fuente conforme exige la licencia Creative Commons Attribution 4.0 bajo la cual se publica.

---

## V. Recursos

- Computadora con acceso a internet y navegador web.
- Cuenta de Google para el uso de Google Colab y cuenta de GitHub.
- Notebook LAB-D4-AEI-PSAYAN-2026-02.ipynb.
- Conjunto de datos Online Retail II, del UCI Machine Learning Repository, alojado en el repositorio del curso.
- Librerías Polars, DuckDB y Plotly.

---

## VI. Metodología para el desarrollo de la tarea

El laboratorio se desarrolla en los equipos del proyecto integrador. Cada estudiante ejecuta el notebook, completa el código solicitado y registra sus respuestas en las celdas indicadas. Las respuestas de la guía se redactan de forma individual, aunque el análisis se discuta en equipo.

Entregable: notebook ejecutado y depositado en el repositorio del equipo, dentro de la carpeta semana04/.

---

## VII. Procedimiento

---

## Actividad 1 — Revisión de conceptos

Complete la tabla siguiente con definiciones elaboradas con sus propias palabras. No se admite la transcripción literal del material de clase ni de fuentes externas.

| Concepto | Definición elaborada por el estudiante |
|---|---|
| Analítica diagnóstica | Analiza datos ya ocurridos para ver con qué factores se relaciona un cambio en un resultado, aunque sin llegar a probar la causa. |
| Driver de un resultado | Variable que se mueve junto con el resultado de forma medible; se confirma descomponiendo el resultado y viendo que la suma cuadra. |
| Efecto volumen | Parte del cambio en ingresos que viene solo de vender más o menos unidades, con precios y mezcla fijos. |
| Efecto mezcla | Parte del cambio en ingresos que viene de vender una proporción distinta de productos, aunque el volumen y los precios no cambien. |
| Efecto precio | Parte del cambio en ingresos que viene de cobrar distinto por los mismos productos entre un periodo y otro. |
| Cohorte de clientes | Grupo de clientes que compraron por primera vez en el mismo mes, a los que se sigue según los meses desde esa fecha y no desde el calendario. |
| Tasa de retención | Porcentaje de una cohorte que sigue comprando cada mes después de su primera compra. |
| Segmentación explicativa | Dividir una variación total en sus partes (país, tipo de cliente, etc.) para ver dónde se concentra el cambio y priorizar por peso económico real. |

---

## Actividad 2 — Desarrollo práctico y ejecución

**Caso de estudio: un comercio electrónico británico de artículos de regalo**

La empresa del caso es un minorista en línea con sede en el Reino Unido que vende artículos de regalo y decoración, en su mayor parte a clientes mayoristas que revenden en tiendas pequeñas de Europa, Asia y Oceanía. Su sistema transaccional registra, línea por línea, cada producto facturado entre el 1 de diciembre de 2009 y el 9 de diciembre de 2011: poco más de un millón de registros con el número de factura, el código y la descripción del producto, las unidades, la fecha y hora, el precio unitario, el identificador del cliente y su país. Es un registro rico, pero también un registro sucio: contiene cancelaciones, devoluciones, cargos de envío mezclados con productos, filas duplicadas y casi una de cada cuatro líneas sin cliente identificado. Ninguna de esas imperfecciones es un defecto del conjunto de datos; son exactamente las que aparecen en cualquier sistema transaccional en operación.

El encargo de la sesión nace de una observación que la gerencia comercial lleva a la reunión de resultados. Comparando el año que va de diciembre de 2009 a noviembre de 2010 con el que va de diciembre de 2010 a noviembre de 2011, los ingresos crecieron, pero el número de unidades vendidas cayó, y lo hizo de forma apreciable. La lectura optimista, que se queda en la cifra de ingresos, concluye que el año fue mejor. La lectura pesimista, que se queda en las unidades, concluye que la empresa está perdiendo mercado. Ambas lecturas son descriptivas y ninguna de las dos explica nada. El laboratorio consiste precisamente en pasar del nivel descriptivo al diagnóstico: descomponer esa variación en los factores que la componen, verificar que la descomposición reconcilia exactamente con la cifra observada, examinar el comportamiento de los clientes mediante cohortes, localizar el segmento donde se concentra el movimiento y, finalmente, señalar cuál es el driver dominante y qué decisión corresponde tomar. El estudiante deberá además indicar qué NO demuestra su análisis, porque un diagnóstico que no reconoce sus límites es un diagnóstico que se presta a decisiones equivocadas.

> **Naturaleza de los datos.** A diferencia del laboratorio de la Semana 1, los datos de esta sesión no son sintéticos. Corresponden al conjunto Online Retail II, publicado por el UCI Machine Learning Repository bajo licencia Creative Commons Attribution 4.0 y donado por el Dr. Daqing Chen, de la London South Bank University. Son transacciones reales de una empresa real, y los importes están expresados en libras esterlinas. El repositorio del curso aloja una copia en formato Parquet, idéntica al archivo original salvo por el formato de almacenamiento; el notebook verifica esa identidad comparando la huella SHA-256 antes de utilizarla. El script que reproduce la conversión desde el origen está publicado junto a los datos.

**Resultados de aprendizaje de la sesión**

- Distinguir en la práctica los niveles descriptivo, diagnóstico y causal de un análisis.
- Someter un conjunto de datos real a un control de calidad y declarar por escrito las reglas aplicadas.
- Descomponer una variación de ingresos en efecto volumen, efecto mezcla y efecto precio, y comprobar que reconcilian.
- Construir una matriz de cohortes y leer correctamente sus celdas vacías.
- Segmentar una variación y priorizar por contribución absoluta antes que por variación porcentual.
- Formular una recomendación empresarial acompañada de la cifra que la sustenta y de su limitación.

**Agenda de la sesión**

| Hora | Duración | Bloque de trabajo |
|---|---|---|
| 7:00 – 7:15 | 15 min | Activación conceptual: describir, diagnosticar y demostrar causalidad. |
| 7:15 – 7:40 | 25 min | Descarga, trazabilidad y control de calidad del dataset UCI. |
| 7:40 – 8:05 | 25 min | Construcción de la tabla de transacciones válidas y comparación temporal. |
| 8:05 – 8:25 | 20 min | Puente precio-volumen-mezcla y comprobación de reconciliación. |
| 8:25 – 8:30 | 5 min | Reto 1: la métrica que describe frente a la que explica. |
| 8:30 – 8:45 | 15 min | RECESO. |
| 8:45 – 9:25 | 40 min | Cohortes de clientes, matriz y curvas de retención. |
| 9:25 – 9:55 | 30 min | Segmentación, tablero Plotly y lectura de desviaciones. |
| 9:55 – 10:10 | 15 min | Reto final, decisión empresarial y ticket de salida. |

### BLOQUE 0 — Activación conceptual

*10 minutos*

Antes de escribir una sola línea de código conviene fijar la distinción que organiza toda la sesión. Los tres niveles siguientes responden a preguntas distintas y no son intercambiables: confundirlos es la causa más frecuente de las recomendaciones mal fundadas.

| Nivel | Pregunta que responde | Lo que NO permite afirmar |
|---|---|---|
| Descriptivo | ¿Qué cambió, cuánto, dónde y cuándo? | Por qué cambió. |
| Diagnóstico | ¿Con qué factores medibles se relaciona el cambio? | Que esos factores lo hayan producido. |
| Causal | ¿Qué intervención produjo el efecto? | Nada adicional, pero exige un diseño experimental o cuasi-experimental. |

**Pregunta 1 — Clasificación previa**

Clasifique cada afirmación como descriptiva, diagnóstica o causal. a) «Los ingresos del segundo año fueron superiores en 2.5 %.» b) «La caída de unidades se concentra en los productos que dejaron de ofrecerse.» c) «Retirar esos productos del catálogo provocó la caída de unidades.»

*Respuesta:* a) **Descriptiva.** Solo dice qué pasó y cuánto (+2.5 %), sin relacionarlo con ningún factor ni con una causa.

*Respuesta:* b) **Diagnóstica.** Conecta la caída de unidades con un factor medible de los datos —los productos discontinuados—, pero no dice que ese factor la haya causado, solo que ahí se concentra.

*Respuesta:* c) **Causal.** Dice que una acción (retirar productos) produjo un efecto (la caída). Eso va más allá de lo que este laboratorio puede sostener: no hay un diseño experimental que aísle esa relación de otras explicaciones posibles, como estacionalidad, sustitución por otros productos o pérdida de clientes por motivos ajenos al catálogo.

Ejecute a continuación las dos primeras celdas del notebook, que verifican la versión de Python del entorno e instalan las librerías del curso. Ejecute las celdas siempre en el orden en que aparecen.

## Organización de la sesión — 7:00 p. m. a 10:10 p. m.

**Duración total:** 190 minutos · **Receso:** 15 minutos · **Trabajo efectivo:** 175 minutos.

| Horario | Tiempo | Desarrollo |
|---|---:|---|
| 7:00–7:10 | 10 min | Apertura del caso y planteamiento del problema de diagnóstico. |
| 7:10–7:35 | 25 min | Actividad 1. Revisión de conceptos. |
| 7:35–8:00 | 25 min | Bloque 1. Descarga, trazabilidad y control de calidad. |
| 8:00–8:30 | 30 min | Bloque 2. Transacciones válidas, periodos comparables y puente precio-volumen-mezcla. |
| 8:30–8:45 | 15 min | **RECESO** |
| 8:45–9:15 | 30 min | Bloque 3. Cohortes de clientes y curvas de retención. |
| 9:15–9:35 | 20 min | Bloque 4. Segmentación explicativa, DuckDB y cuadro de drivers. |
| 9:35–10:00 | 25 min | **Reto de aplicación y retroalimentación.** Ejercicios 1 a 5. |
| 10:00–10:10 | 10 min | Decisión empresarial y ticket de salida. |

> **Regla de trabajo:** no avance de bloque sin registrar la interpretación solicitada. El objetivo no es ejecutar celdas, sino convertir datos en evidencia para una decisión.


## Actividad 1 — Revisión de conceptos (25 minutos)

**Propósito.** Establecer con precisión el vocabulario del análisis diagnóstico antes de programar. La descomposición de una variación carece de valor si los efectos que la componen no se distinguen con exactitud.

**Instrucciones.** Complete la tabla con definiciones elaboradas con sus propias palabras. No se admite la reproducción literal de fuentes externas ni de sistemas generativos. La columna de la derecha contiene una pregunta de apoyo: si su definición permite responderla, la definición es suficiente; si no lo permite, corríjala antes de continuar.

**Evidencia esperada.** Tabla completa con las definiciones registradas.

| Concepto | Definición elaborada por el estudiante | Pregunta de apoyo |
|---|---|---|
| Analítica diagnóstica | Es el análisis de datos históricos para entender con qué variables medibles se asocia un cambio en un resultado; va un paso más allá de simplemente describir lo ocurrido, pero todavía no permite afirmar qué lo provocó. | ¿Qué la distingue de la descriptiva y qué sigue sin poder afirmar? |
| Driver de un resultado | Es una variable medible cuyos movimientos acompañan de manera sistemática al cambio del resultado analizado; se valida como tal descomponiendo el resultado en sus partes y comprobando que la suma cuadra con la variación observada. | ¿Cómo se comprueba que una variable es driver y no una simple correlación? |
| Efecto volumen | Es la porción del cambio en ingresos que se explica solo por haber vendido más o menos unidades en total, dejando fijos tanto los precios como la mezcla de productos del periodo base. | Si el precio no cambia y los ingresos caen, ¿qué indica el efecto volumen? |
| Efecto mezcla | Es la porción del cambio en ingresos que ocurre porque varía la proporción que cada producto representa dentro del total de unidades vendidas, incluso si el volumen total y los precios no cambian. | ¿Puede aumentar el ingreso sin vender más unidades ni subir precios? Explique. |
| Efecto precio | Es la porción del cambio en ingresos que se debe a que los mismos productos se cobraron a un precio efectivo distinto entre un periodo y otro, manteniendo fijos el volumen y la mezcla del periodo final. | ¿Por qué debe calcularse sobre un catálogo común a ambos periodos? |
| Cohorte de clientes | Es el conjunto de clientes que realizaron su primera compra dentro del mismo mes, a los que se les sigue el comportamiento posterior contando los meses desde ese punto de partida común y no desde una fecha fija del calendario. | ¿Por qué la antigüedad se mide desde el mes de la primera compra y no desde enero? |
| Tasa de retención | Es el porcentaje de clientes de una cohorte que sigue comprando en cada mes posterior a su primera compra, calculado siempre respecto al tamaño con el que nació esa misma cohorte. | ¿Qué significa una celda vacía en la matriz de retención de una cohorte reciente? |
| Segmentación explicativa | Es dividir una variación total en las partes que la componen (por país, tipo de cliente, etc.) para identificar en qué segmento específico se concentra el cambio, priorizando según el peso económico real y no solo según el porcentaje de variación. | ¿Por qué el segmento de mayor caída porcentual puede no ser el prioritario? |


**Criterio de cierre.** No se avanza al desarrollo práctico mientras existan conceptos sin definición registrada.


In [ ]:
# Verificamos con qué versión de Python trabaja el entorno.
import sys, platform

print("Python  :", sys.version.split()[0])
print("Sistema :", platform.system())

Python  : 3.13.15
Sistema : Linux


In [ ]:
# Instalamos las librerías que Colab no trae por defecto.
# Fijamos la versión a propósito: así el notebook seguirá funcionando en diciembre.
!pip install -q polars==1.17.1 duckdb==1.1.3

import polars as pl
import duckdb
import plotly.graph_objects as go
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_fmt_float("full")   # cifras completas, sin notación científica

print("polars :", pl.__version__)
print("duckdb :", duckdb.__version__)
print("\nEntorno listo.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 36.9 MB/s eta 0:00:00
polars : 1.17.1
duckdb : 1.1.3

Entorno listo.


### BLOQUE 1 — Descarga, trazabilidad y control de calidad

*15 minutos*

Un análisis solo vale lo que vale su fuente. Este bloque descarga el conjunto de datos, comprueba que sea exactamente el publicado y levanta un inventario de sus problemas de calidad. Obsérvese el orden: primero se mide cuánto hay de cada problema y recién después se decide qué hacer con él. Descartar filas antes de saber cuántas son ni cuánto dinero mueven es una forma silenciosa de alterar el resultado.

In [ ]:
# Descargamos el conjunto de datos y verificamos que sea exactamente el
# publicado. La huella SHA-256 es la constancia de que nadie lo alteró:
# si un solo valor cambiara, la huella sería distinta.
import hashlib, pathlib, urllib.request

URL = ("https://raw.githubusercontent.com/Rociosayan/"
       "analitica-empresarial-integrada/main/semana04/datos/"
       "online_retail_II.parquet")
HUELLA_PUBLICADA = "8f64c20d17d38ab02c0dfddda323574e0ee8c34e719df6ea574d16a6e3678e96"

ARCHIVO = pathlib.Path("online_retail_II.parquet")
if not ARCHIVO.exists():
    urllib.request.urlretrieve(URL, ARCHIVO)

huella = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
print("Tamaño        :", f"{ARCHIVO.stat().st_size:,} bytes")
print("SHA-256       :", huella)
print("¿Coincide?    :",
      "sí" if huella == HUELLA_PUBLICADA else "NO — avise a la docente")

Tamaño        : 7,283,898 bytes
SHA-256       : 8f64c20d17d38ab02c0dfddda323574e0ee8c34e719df6ea574d16a6e3678e96
¿Coincide?    : sí


In [ ]:
# Ficha de trazabilidad. Todo análisis serio empieza declarando de dónde
# salieron los datos: sin origen verificable, ninguna cifra es defendible.
FICHA = {
    "Conjunto":  "Online Retail II",
    "Origen":    "UCI Machine Learning Repository, conjunto n.° 502",
    "Enlace":    "https://archive.ics.uci.edu/dataset/502/online+retail+ii",
    "Donante":   "Dr. Daqing Chen, London South Bank University",
    "Licencia":  "Creative Commons Attribution 4.0 (CC BY 4.0)",
    "Cobertura": "01/12/2009 a 09/12/2011",
    "Moneda":    "Libra esterlina (GBP)",
}
for campo, valor in FICHA.items():
    print(f"{campo:<10}: {valor}")

Conjunto  : Online Retail II
Origen    : UCI Machine Learning Repository, conjunto n.° 502
Enlace    : https://archive.ics.uci.edu/dataset/502/online+retail+ii
Donante   : Dr. Daqing Chen, London South Bank University
Licencia  : Creative Commons Attribution 4.0 (CC BY 4.0)
Cobertura : 01/12/2009 a 09/12/2011
Moneda    : Libra esterlina (GBP)


In [ ]:
# Cargamos los datos y renombramos las columnas al castellano. El nombre
# original "Customer ID" lleva un espacio, lo que obliga a escribirlo entre
# comillas en cada consulta; renombrarlo ahora evita ese estorbo después.
crudo = pl.read_parquet(ARCHIVO).rename({
    "Invoice":     "factura",
    "StockCode":   "codigo",
    "Description": "descripcion",
    "Quantity":    "cantidad",
    "InvoiceDate": "fecha",
    "Price":       "precio",
    "Customer ID": "cliente_id",
    "Country":     "pais",
})

print("Filas    :", f"{crudo.height:,}")
print("Columnas :", crudo.width)
print("Periodo  :", crudo["fecha"].min(), "a", crudo["fecha"].max())
print()
print(crudo.head(5))

Filas    : 1,067,371
Columnas : 8
Periodo  : 2009-12-01 07:45:00 a 2011-12-09 12:50:00

shape: (5, 8)
┌─────────┬────────┬─────────────────────────────────┬──────────┬─────────────────────┬────────┬────────────┬────────────────┐
│ factura ┆ codigo ┆ descripcion                     ┆ cantidad ┆ fecha               ┆ precio ┆ cliente_id ┆ pais           │
│ ---     ┆ ---    ┆ ---                             ┆ ---      ┆ ---                 ┆ ---    ┆ ---        ┆ ---            │
│ str     ┆ str    ┆ str                             ┆ i64      ┆ datetime[ns]        ┆ f64    ┆ f64        ┆ str            │
╞═════════╪════════╪═════════════════════════════════╪══════════╪═════════════════════╪════════╪════════════╪════════════════╡
│ 489434  ┆ 85048  ┆ 15CM CHRISTMAS GLASS BALL 20 L… ┆ 12       ┆ 2009-12-01 07:45:00 ┆ 6.95   ┆ 13085      ┆ United Kingdom │
│ 489434  ┆ 79323P ┆ PINK CHERRY LIGHTS              ┆ 12       ┆ 2009-12-01 07:45:00 ┆ 6.75   ┆ 13085      ┆ United Kingdom │
│ 489434 

In [ ]:
# Inventario de problemas de calidad. Todavía no corregimos nada: primero
# medimos cuánto hay de cada cosa, porque la magnitud del problema determina
# si conviene descartarlo, corregirlo o dejarlo declarado.
SERVICIOS = (r"(?i)^(POST|DOT|M|C2|C3|D|S|B|BANK CHARGES|ADJUST\d?"
             r"|AMAZONFEE|PADS|CRUK|TEST\d+|GIFT|gift_.*)$")

diagnostico = {
    "Filas totales":
        crudo.height,
    "Filas duplicadas exactas":
        crudo.height - crudo.unique().height,
    "Cancelaciones (factura que empieza con C)":
        crudo.filter(pl.col("factura").str.starts_with("C")).height,
    "Cantidad menor o igual a cero":
        crudo.filter(pl.col("cantidad") <= 0).height,
    "Precio menor o igual a cero":
        crudo.filter(pl.col("precio") <= 0).height,
    "Cliente sin identificar":
        crudo.filter(pl.col("cliente_id").is_null()).height,
    "Servicios y ajustes, no productos":
        crudo.filter(pl.col("codigo").str.contains(SERVICIOS)).height,
}
for problema, cantidad in diagnostico.items():
    print(f"{problema:<44} {cantidad:>9,}   "
          f"({100 * cantidad / crudo.height:5.2f} %)")

Filas totales                                1,067,371   (100.00 %)
Filas duplicadas exactas                        34,335   ( 3.22 %)
Cancelaciones (factura que empieza con C)       19,494   ( 1.83 %)
Cantidad menor o igual a cero                   22,950   ( 2.15 %)
Precio menor o igual a cero                      6,207   ( 0.58 %)
Cliente sin identificar                        243,007   (22.77 %)
Servicios y ajustes, no productos                5,932   ( 0.56 %)


In [ ]:
# ¿Qué son esos códigos que no son productos? Conviene mirarlos antes de
# descartarlos: si uno de ellos moviera mucho dinero, excluirlo sin avisar
# distorsionaría el diagnóstico.
(crudo
 .filter(pl.col("codigo").str.contains(SERVICIOS))
 .group_by("codigo")
 .agg(pl.len().alias("lineas"),
      pl.col("descripcion").first().alias("descripcion"))
 .sort("lineas", descending=True)
 .head(10))

codigo,lineas,descripcion
str,u32,str
"""POST""",2122,"""POSTAGE"""
"""DOT""",1446,"""DOTCOM POSTAGE"""
"""M""",1421,"""Manual"""
"""C2""",282,"""CARRIAGE"""
"""D""",177,"""Discount"""
"""S""",104,"""SAMPLES"""
"""BANK CHARGES""",102,""" Bank Charges"""
"""ADJUST""",67,"""Adjustment by john on 26/01/20…"
"""AMAZONFEE""",43,"""AMAZON FEE"""


**Pregunta 2 — Decisiones de calidad**

De los problemas listados, el de mayor volumen es el de las líneas sin cliente identificado, cerca del 23 % del total. a) ¿Por qué esas líneas sí pueden usarse para calcular ingresos? b) ¿Por qué no pueden usarse para construir cohortes? c) ¿Qué distorsión introduciría descartarlas desde el inicio?

*Respuesta:* a) Esas líneas sí sirven para calcular ingresos porque el monto de cada venta sale de multiplicar cantidad por precio, y ambas columnas están completas aunque no se sepa quién compró. La venta ya ocurrió y puede verificarse sin necesidad de conocer al cliente.

*Respuesta:* b) Para armar cohortes sí hace falta el identificador, porque una cohorte agrupa a los clientes según el mes de su primera compra. Sin ese dato no hay forma de saber si dos líneas pertenecen a la misma persona ni de seguir su historial de compras en el tiempo.

*Respuesta:* c) Si se descartaran de entrada, los ingresos totales quedarían subestimados en cerca de 23 %, mostrando un negocio más pequeño de lo real. Aquí el problema es de trazabilidad del cliente, no de que la venta sea inválida, así que eliminarlas de plano mezclaría dos cuestiones distintas.

### BLOQUE 2 — Transacciones válidas y comparación temporal

*15 minutos*

Con el inventario a la vista se construye la tabla de transacciones válidas aplicando cuatro reglas explícitas. Cada regla es una definición de qué cuenta como venta, y esas definiciones tendrán que declararse en el informe: dos analistas que apliquen reglas distintas obtendrán cifras distintas sin que ninguno se haya equivocado.


In [ ]:
# Tabla de transacciones válidas. Cada filtro responde a una definición
# explícita de qué cuenta como venta; esas definiciones son las que después
# habrá que declarar en el informe.
validas = (
    crudo
    # 1. una fila repetida no representa una venta adicional
    .unique()
    # 2. las cancelaciones no son ventas
    .filter(~pl.col("factura").str.starts_with("C"))
    # 3. sin unidades o sin precio no hay ingreso que medir
    .filter((pl.col("cantidad") > 0) & (pl.col("precio") > 0))
    # 4. los envíos y ajustes no son producto vendido
    .filter(~pl.col("codigo").str.contains(SERVICIOS))
    .with_columns(
        (pl.col("cantidad") * pl.col("precio")).alias("ingreso"),
        pl.col("fecha").dt.date().alias("dia"),
    )
)

print(f"Filas de partida : {crudo.height:>9,}")
print(f"Filas válidas    : {validas.height:>9,}")
print(f"Descartadas      : {crudo.height - validas.height:>9,} "
      f"({100 * (crudo.height - validas.height) / crudo.height:.2f} %)")

Filas de partida : 1,067,371
Filas válidas    : 1,003,340
Descartadas      :    64,031 (6.00 %)


A continuación se delimitan los dos periodos de comparación. Se toman doce meses exactos en cada caso, de diciembre a noviembre. La razón es que el conjunto termina el 9 de diciembre de 2011: incluir ese mes incompleto compararía doce meses contra once meses y nueve días, y fabricaría una caída que no ocurrió.

In [ ]:
# Definimos dos periodos de doce meses exactos. Comparar años calendario
# incompletos es el error más común del análisis diagnóstico: diciembre de
# 2011 está truncado el día 9, de modo que incluirlo fabricaría una caída
# que no existe.
INICIO_P0, FIN_P0 = pl.date(2009, 12, 1), pl.date(2010, 11, 30)
INICIO_P1, FIN_P1 = pl.date(2010, 12, 1), pl.date(2011, 11, 30)

validas = validas.with_columns(
    pl.when((pl.col("dia") >= INICIO_P0) & (pl.col("dia") <= FIN_P0)).then(pl.lit("P0"))
      .when((pl.col("dia") >= INICIO_P1) & (pl.col("dia") <= FIN_P1)).then(pl.lit("P1"))
      .otherwise(None)
      .alias("periodo")
)

fuera = validas.filter(pl.col("periodo").is_null()).height
print(f"P0: dic-2009 a nov-2010     P1: dic-2010 a nov-2011")
print(f"Filas fuera de ambos periodos (diciembre de 2011 incompleto): {fuera:,}")

comparables = validas.filter(pl.col("periodo").is_not_null())
print(f"Filas comparables: {comparables.height:,}")

P0: dic-2009 a nov-2010     P1: dic-2010 a nov-2011
Filas fuera de ambos periodos (diciembre de 2011 incompleto): 24,753
Filas comparables: 978,587


In [ ]:
# Comparación temporal. Este es el nivel descriptivo: dice qué cambió y
# cuánto, pero todavía no explica por qué.
resumen = (
    comparables
    .group_by("periodo")
    .agg(
        pl.col("ingreso").sum().alias("ingresos"),
        pl.col("cantidad").sum().alias("unidades"),
        pl.col("factura").n_unique().alias("facturas"),
        pl.col("cliente_id").n_unique().alias("clientes"),
        pl.col("codigo").n_unique().alias("productos"),
    )
    .sort("periodo")
)

base, final = resumen.row(0, named=True), resumen.row(1, named=True)
print(f"{'Indicador':<12}{'P0':>16}{'P1':>16}{'Variación':>16}{'%':>10}")
for indicador in ["ingresos", "unidades", "facturas", "clientes", "productos"]:
    a, b = base[indicador], final[indicador]
    print(f"{indicador:<12}{a:>16,.0f}{b:>16,.0f}"
          f"{b - a:>16,.0f}{100 * (b - a) / a:>9.2f}%")

ticket_p0 = base["ingresos"] / base["facturas"]
ticket_p1 = final["ingresos"] / final["facturas"]
print(f"\nTicket promedio  P0: £ {ticket_p0:,.2f}   P1: £ {ticket_p1:,.2f}")

Indicador                 P0              P1       Variación         %
ingresos           9,396,642       9,632,728         236,086     2.51%
unidades           5,626,660       5,248,737        -377,923    -6.72%
facturas              19,743          18,957            -786    -3.98%
clientes               4,240           4,294              54     1.27%
productos              4,220           3,904            -316    -7.49%

Ticket promedio  P0: £ 475.95   P1: £ 508.14


In [ ]:
# Serie mensual de ingresos. El gráfico permite ver si la variación anual
# proviene de un cambio sostenido o de unos pocos meses atípicos.
mensual = (
    comparables
    .with_columns(pl.col("fecha").dt.truncate("1mo").dt.date().alias("mes"))
    .group_by(["periodo", "mes"])
    .agg(pl.col("ingreso").sum().alias("ingresos"))
    .sort("mes")
)

figura = px.line(mensual.to_pandas(), x="mes", y="ingresos", color="periodo",
                 markers=True, labels={"mes": "Mes", "ingresos": "Ingresos (£)",
                                       "periodo": "Periodo"})
figura.update_layout(title="Ingresos mensuales por periodo", height=420,
                     hovermode="x unified")
figura.show()

**Pregunta 3 — La paradoja**

Observe el cuadro comparativo: los ingresos y las unidades se mueven en sentidos opuestos. a) Enuncie con sus palabras qué situación de negocio podría producir ese resultado. b) ¿El ticket promedio subió porque subieron los precios? Justifique por qué esa conclusión todavía no puede sostenerse con la información disponible hasta aquí.

*Respuesta:* a) Esto puede pasar cuando la empresa vende menos unidades en total, pero entre lo que sí vende predominan artículos de mayor valor unitario, o cuando el precio efectivo sube lo suficiente como para compensar de sobra la caída en volumen. En cualquiera de los dos casos, el ingreso termina creciendo aunque se muevan menos productos.

*Respuesta:* b) Todavía no. El ticket promedio pasa de £475.95 a £508.14, pero ese número junta dos cosas al mismo tiempo: el efecto de vender una mezcla distinta de productos y el efecto de vender a precios distintos. Para separar cuánto corresponde a cada uno hace falta la descomposición precio-volumen-mezcla del bloque siguiente.

*Respuesta:*

### BLOQUE 3 — Puente precio-volumen-mezcla

*15 minutos*

Este es el núcleo del laboratorio. La variación de ingresos se atribuye a tres efectos independientes, calculados sobre los productos presentes en ambos periodos. El efecto volumen recoge el cambio por vender más o menos unidades; el efecto mezcla, el cambio por vender una composición distinta de productos; el efecto precio, el cambio por vender a precios distintos. La suma de los tres debe reconciliar exactamente con la variación observada.

Antes de aplicar las fórmulas hay que resolver una dificultad que el material teórico no aborda y que los datos reales imponen: el catálogo no es el mismo en los dos periodos. Hay productos que dejaron de venderse y productos que aparecieron, y estos no tienen precio de comparación. Se los separa y se los cuantifica por su cuenta, de modo que el puente quede completo y siga reconciliando.


In [ ]:
# Para descomponer la variación necesitamos, en cada periodo y por producto,
# las unidades vendidas y el precio efectivo. El precio efectivo no es el de
# lista: es el ingreso dividido entre las unidades, es decir, el precio al que
# realmente se vendió considerando todas sus operaciones.
por_producto = (
    comparables
    .group_by(["periodo", "codigo"])
    .agg(pl.col("cantidad").sum().alias("q"),
         pl.col("ingreso").sum().alias("r"))
    .with_columns((pl.col("r") / pl.col("q")).alias("p"))
)

p0 = por_producto.filter(pl.col("periodo") == "P0").drop("periodo")
p1 = por_producto.filter(pl.col("periodo") == "P1").drop("periodo")
print("Productos en P0:", p0.height, "   Productos en P1:", p1.height)

Productos en P0: 4220    Productos en P1: 3904


In [ ]:
# El catálogo cambia entre periodos: hay productos que dejaron de venderse y
# productos que aparecieron. La descomposición precio-volumen-mezcla solo
# puede aplicarse a los que existen en ambos, porque los demás no tienen
# precio de comparación. Los separamos y los cuantificamos aparte: ocultarlos
# rompería la reconciliación.
comun          = p0.join(p1, on="codigo", suffix="_1")
discontinuados = p0.join(p1, on="codigo", how="anti")
nuevos         = p1.join(p0, on="codigo", how="anti")

print(f"Catálogo común  : {comun.height:>5} productos")
print(f"Discontinuados  : {discontinuados.height:>5} productos   "
      f"£ {discontinuados['r'].sum():>14,.2f}")
print(f"Nuevos en P1    : {nuevos.height:>5} productos   "
      f"£ {nuevos['r'].sum():>14,.2f}")

Catálogo común  :  3230 productos
Discontinuados  :   990 productos   £     704,357.16
Nuevos en P1    :   674 productos   £   2,046,903.92


In [ ]:
# Descomposición precio-volumen-mezcla sobre el catálogo común.
#
#   E_volumen = (Q1 - Q0) * suma( s0i * p0i )
#   E_mezcla  = Q1 * suma( (s1i - s0i) * p0i )
#   E_precio  = Q1 * suma( s1i * (p1i - p0i) )
#
# donde Q es el volumen total del periodo, s la participación del producto en
# ese volumen y p su precio efectivo. El orden importa: cada efecto se calcula
# manteniendo constantes los factores que ya se aislaron.
Q0 = comun["q"].sum()
Q1 = comun["q_1"].sum()

c = comun.with_columns((pl.col("q") / Q0).alias("s0"),
                       (pl.col("q_1") / Q1).alias("s1"))

efecto_volumen = (Q1 - Q0) * (c["s0"] * c["p"]).sum()
# El efecto mezcla es el más sutil de los tres: aísla el cambio de composición
# del catálogo manteniendo constantes el volumen total y los precios del
# periodo base. Un catalogo que se desplaza hacia productos mas baratos reduce
# el ingreso aunque no cambien ni el volumen total ni los precios.
efecto_mezcla  = Q1 * ((c["s1"] - c["s0"]) * c["p"]).sum()
efecto_precio  = Q1 * (c["s1"] * (c["p_1"] - c["p"])).sum()

variacion_comun = c["r_1"].sum() - c["r"].sum()
suma_efectos    = efecto_volumen + efecto_mezcla + efecto_precio

print(f"Efecto volumen : £ {efecto_volumen:>14,.2f}")
print(f"Efecto mezcla  : £ {efecto_mezcla:>14,.2f}")
print(f"Efecto precio  : £ {efecto_precio:>14,.2f}")
print(f"{'-' * 34}")
print(f"Suma           : £ {suma_efectos:>14,.2f}")
print(f"Variación real : £ {variacion_comun:>14,.2f}")
print(f"Diferencia     : £ {variacion_comun - suma_efectos:>14,.2f}")


Efecto volumen : £  -1,568,093.05
Efecto mezcla  : £     577,550.82
Efecto precio  : £    -115,918.44
----------------------------------
Suma           : £  -1,106,460.66
Variación real : £  -1,106,460.66
Diferencia     : £           0.00


In [ ]:
# Comprobación obligatoria. Si los efectos no reconcilian con la variación
# observada, existe un error metodológico y el diagnóstico no debe publicarse.
# Preferimos que el notebook se detenga aquí antes que entregar una cifra falsa.
assert abs(variacion_comun - suma_efectos) < 0.01, "Los efectos no reconcilian"

variacion_total = p1["r"].sum() - p0["r"].sum()
reconstruida    = variacion_comun + nuevos["r"].sum() - discontinuados["r"].sum()
assert abs(variacion_total - reconstruida) < 0.01, "El puente completo no reconcilia"

print("Reconciliación verificada.")
print(f"\nVariación total de ingresos : £ {variacion_total:,.2f}")
print(f"  Catálogo común            : £ {variacion_comun:,.2f}")
print(f"    · efecto volumen        : £ {efecto_volumen:,.2f}")
print(f"    · efecto mezcla         : £ {efecto_mezcla:,.2f}")
print(f"    · efecto precio         : £ {efecto_precio:,.2f}")
print(f"  Productos nuevos          : £ {nuevos['r'].sum():,.2f}")
print(f"  Productos discontinuados  : £ {-discontinuados['r'].sum():,.2f}")

Reconciliación verificada.

Variación total de ingresos : £ 236,086.10
  Catálogo común            : £ -1,106,460.66
    · efecto volumen        : £ -1,568,093.05
    · efecto mezcla         : £ 577,550.82
    · efecto precio         : £ -115,918.44
  Productos nuevos          : £ 2,046,903.92
  Productos discontinuados  : £ -704,357.16


In [ ]:
# Gráfico de cascada. Cada barra parte del subtotal anterior, de modo que se
# ve cómo se pasa del ingreso del periodo base al del periodo final.
figura = go.Figure(go.Waterfall(
    orientation="v",
    measure=["absolute", "relative", "relative", "relative",
             "relative", "relative", "total"],
    x=["Ingresos P0", "Volumen", "Mezcla", "Precio",
       "Productos nuevos", "Discontinuados", "Ingresos P1"],
    y=[p0["r"].sum(), efecto_volumen, efecto_mezcla, efecto_precio,
       nuevos["r"].sum(), -discontinuados["r"].sum(), p1["r"].sum()],
    text=[f"£ {v:,.0f}" for v in
          [p0["r"].sum(), efecto_volumen, efecto_mezcla, efecto_precio,
           nuevos["r"].sum(), -discontinuados["r"].sum(), p1["r"].sum()]],
    textposition="outside",
    connector={"line": {"color": "#9AA7B4"}},
))
figura.update_layout(title="Puente de ingresos: de P0 a P1",
                     yaxis_title="Ingresos (£)", height=520, showlegend=False)
figura.show()

**Pregunta 4 — Lectura del puente**

a) ¿Cuál de los cinco componentes del puente tiene mayor magnitud absoluta y de cuánto es? b) El efecto mezcla resultó positivo. Explique qué significa eso en términos de qué productos se vendieron. c) ¿Por qué la comprobación de reconciliación es obligatoria y no un adorno del análisis?

*Respuesta:* a) De los cinco componentes del puente, el que más pesa en valor absoluto es "productos nuevos", con £2,046,903.92 (40.8 % del peso relativo total); muy cerca detrás aparece el efecto volumen del catálogo común, con £-1,568,093.05 (31.3 %).

*Respuesta:* b) Que el efecto mezcla haya salido positivo (£577,550.82) significa que, dentro del catálogo vendido en ambos periodos, las unidades se concentraron más en productos de precio efectivo más alto. Dicho de otra forma: cayó el volumen total, pero lo que sí se vendió se inclinó hacia los artículos más caros de ese mismo catálogo.

*Respuesta:* c) Porque si no se verifica que la suma de los efectos calza, al centavo, con la variación observada, no hay garantía de que la fórmula se haya aplicado bien. Un error de cálculo podría pasar inadvertido y terminaría entregándose a la gerencia una cifra que no cuadra con la realidad, lo que invalidaría cualquier decisión tomada sobre esa base.

### RETO 1 — La métrica que describe frente a la que explica

*5 minutos*

Complete la celda siguiente. El notebook verificará su respuesta contra las cifras que usted mismo calculó.


**Pregunta 5 — Métrica de vanidad**

La cifra «los ingresos crecieron 2.5 %» es correcta y, sin embargo, es una métrica de vanidad en este caso. a) Explique por qué. b) Proponga una métrica alternativa, calculable con los datos de este laboratorio, que sí sea accionable para la gerencia comercial.

*Respuesta:* a) Es una métrica de vanidad porque, vista sola, da la impresión de que "todo va bien" sin explicar qué la produjo. Ese mismo 2.51 % de crecimiento en ingresos convive con una caída de 6.72 % en unidades vendidas y de 7.49 % en la cantidad de productos distintos comercializados: son señales de que, en volumen real de actividad, el negocio se está reduciendo, algo que el ingreso por sí solo no muestra.

*Respuesta:* b) Propongo el ingreso por cliente separado según su condición: en P1, un cliente recurrente generó en promedio £2,502.51, frente a £913.70 de uno nuevo. Cruzar esta cifra con la tasa de retención por cohorte a una antigüedad fija (por ejemplo, el mes 6) permite saber si el crecimiento del ingreso viene de retener y desarrollar la base existente o de reponerla constantemente con clientes nuevos, dos escenarios con implicancias muy distintas para la sostenibilidad del negocio.

*Respuesta:*

### BLOQUE 4 — Cohortes y curvas de retención

*15 minutos*

El puente explicó la variación desde el lado del producto. Este bloque la examina desde el lado del cliente. Una cohorte agrupa a los clientes según el mes de su primera compra, y la antigüedad se mide desde ese mes y no desde enero. Esa es toda la diferencia: permite comparar grupos que nacieron en fechas distintas, porque los pone a todos en el mismo punto de partida.


In [ ]:
# Construcción de cohortes. Una cohorte agrupa a los clientes por el mes de su
# primera compra. La antigüedad se mide desde ese mes, no desde enero: por eso
# las cohortes pueden compararse entre sí aunque hayan nacido en fechas
# distintas.
clientes = (
    validas
    .filter(pl.col("cliente_id").is_not_null())
    .with_columns(pl.col("fecha").dt.truncate("1mo").dt.date().alias("mes"))
)

clientes = clientes.with_columns(
    pl.col("mes").min().over("cliente_id").alias("cohorte")
)

clientes = clientes.with_columns(
    ((pl.col("mes").dt.year() - pl.col("cohorte").dt.year()) * 12
     + (pl.col("mes").dt.month() - pl.col("cohorte").dt.month())).alias("antiguedad")
)

print("Clientes identificados :", f"{clientes['cliente_id'].n_unique():,}")
print("Cohortes mensuales     :", clientes["cohorte"].n_unique())
print(clientes.select("cliente_id", "mes", "cohorte", "antiguedad").head(5))

Clientes identificados : 5,852
Cohortes mensuales     : 25
shape: (5, 4)
┌────────────┬────────────┬────────────┬────────────┐
│ cliente_id ┆ mes        ┆ cohorte    ┆ antiguedad │
│ ---        ┆ ---        ┆ ---        ┆ ---        │
│ f64        ┆ date       ┆ date       ┆ i32        │
╞════════════╪════════════╪════════════╪════════════╡
│ 12779      ┆ 2011-11-01 ┆ 2009-12-01 ┆ 23         │
│ 14499      ┆ 2010-11-01 ┆ 2010-08-01 ┆ 3          │
│ 15827      ┆ 2010-02-01 ┆ 2010-01-01 ┆ 1          │
│ 13269      ┆ 2011-09-01 ┆ 2009-12-01 ┆ 21         │
│ 17377      ┆ 2010-05-01 ┆ 2009-12-01 ┆ 5          │
└────────────┴────────────┴────────────┴────────────┘


In [ ]:
# Tasa de retención: de los clientes que estrenaron una cohorte, qué
# proporción seguía comprando en cada mes de antigüedad.
activos = (
    clientes
    .group_by(["cohorte", "antiguedad"])
    .agg(pl.col("cliente_id").n_unique().alias("activos"))
)

tamano_inicial = (
    activos
    .filter(pl.col("antiguedad") == 0)
    .select("cohorte", pl.col("activos").alias("inicial"))
)

retencion = (
    activos
    .join(tamano_inicial, on="cohorte")
    .with_columns((100 * pl.col("activos") / pl.col("inicial")).alias("tasa"))
    .sort(["cohorte", "antiguedad"])
)

matriz = (
    retencion
    .pivot(values="tasa", index="cohorte", on="antiguedad", aggregate_function="first")
    .sort("cohorte")
)
print(matriz.select(["cohorte"] + [str(i) for i in range(6)]))

shape: (25, 7)
┌────────────┬─────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┐
│ cohorte    ┆ 0   ┆ 1                  ┆ 2                  ┆ 3                  ┆ 4                  ┆ 5                  │
│ ---        ┆ --- ┆ ---                ┆ ---                ┆ ---                ┆ ---                ┆ ---                │
│ date       ┆ f64 ┆ f64                ┆ f64                ┆ f64                ┆ f64                ┆ f64                │
╞════════════╪═════╪════════════════════╪════════════════════╪════════════════════╪════════════════════╪════════════════════╡
│ 2009-12-01 ┆ 100 ┆ 35.01577287066246  ┆ 33.333333333333336 ┆ 42.48159831756046  ┆ 37.85488958990536  ┆ 35.96214511041009  │
│ 2010-01-01 ┆ 100 ┆ 21.467391304347824 ┆ 32.06521739130435  ┆ 31.52173913043478  ┆ 27.17391304347826  ┆ 30.97826086956522  │
│ 2010-02-01 ┆ 100 ┆ 23.466666666666665 ┆ 22.666666666666668 ┆ 29.333333333333332 ┆ 24.533333333333335 

In [ ]:
# Matriz de retención. Las celdas vacías de las cohortes recientes no son
# retención cero: son meses que esa cohorte todavía no alcanzó. Confundir
# ambas cosas es uno de los errores clásicos del análisis de cohortes.
columnas = [str(i) for i in range(13)]
tabla = matriz.select(["cohorte"] + [c for c in columnas if c in matriz.columns])

figura = px.imshow(
    tabla.drop("cohorte").to_numpy(),
    x=[f"Mes {c}" for c in tabla.columns[1:]],
    y=[str(f) for f in tabla["cohorte"].to_list()],
    color_continuous_scale="Blues",
    aspect="auto",
    labels={"color": "Retención (%)"},
    text_auto=".0f",
)
figura.update_layout(title="Matriz de retención por cohorte mensual",
                     xaxis_title="Antigüedad",
                     yaxis_title="Cohorte (mes de primera compra)",
                     height=620)
figura.show()

In [ ]:
# Curvas de retención de las cohortes que ya cumplieron seis meses. Todas
# arrancan en 100 % por definición: lo informativo es la pendiente.
maduras = (
    retencion
    .join(retencion.group_by("cohorte").agg(pl.col("antiguedad").max().alias("tope")),
          on="cohorte")
    .filter((pl.col("tope") >= 6) & (pl.col("antiguedad") <= 6))
    .with_columns(pl.col("cohorte").cast(pl.Utf8))
)

figura = px.line(maduras.to_pandas(), x="antiguedad", y="tasa", color="cohorte",
                 markers=True,
                 labels={"antiguedad": "Meses desde la primera compra",
                         "tasa": "Retención (%)", "cohorte": "Cohorte"})
figura.update_layout(title="Curvas de retención por cohorte", height=480)
figura.show()

> **Advertencia de lectura.** Las celdas vacías en las cohortes más recientes no significan retención cero: significan que esa cohorte todavía no alcanzó ese mes de antigüedad. Comparar una cohorte de dos meses con una de doce usando el mismo mes calendario produce una comparación inválida, y es uno de los errores que el material de la semana señala expresamente.

**Pregunta 6 — Lectura de la matriz**

a) Compare la retención en el mes 1 de las cohortes nacidas en el primer año con las del segundo. ¿Qué observa? b) Elija dos cohortes con al menos seis meses de antigüedad y compárelas correctamente; indique cuál retiene mejor y con qué cifras lo sostiene. c) ¿Por qué no puede concluirse de esta matriz que la calidad del servicio empeoró?

*Respuesta:* a) En el mes 1, las cohortes nacidas entre diciembre de 2009 y noviembre de 2010 retienen en promedio 21.08 %, un valor casi idéntico al 20.90 % de las cohortes nacidas entre diciembre de 2010 y noviembre de 2011. No hay un deterioro claro de un año a otro en ese punto de la curva: la retención al mes 1 se mantiene bastante estable.

*Respuesta:* b) Tomando dos cohortes con al menos seis meses de antigüedad: la de diciembre de 2009 retiene 37.64 % de sus clientes en el mes 6, frente a apenas 8.33 % de la cohorte de junio de 2011 en ese mismo punto. La diferencia es amplia; la cohorte de diciembre de 2009 retiene mucho mejor.

*Respuesta:* c) Porque la matriz solo indica cuántos clientes de cada cohorte siguieron comprando, no las razones por las que dejaron de hacerlo. Esa caída podría explicarse por saturación del mercado, estacionalidad, el propio ciclo de vida del cliente mayorista o factores comerciales que nada tienen que ver con el servicio; sin datos de satisfacción, quejas o calidad de atención, atribuirla específicamente al servicio sería una conclusión causal que los datos no permiten sostener.

### BLOQUE 5 — Segmentación, driver dominante y tablero

*10 minutos*

La segmentación localiza dónde se concentra el movimiento. Se examinan dos dimensiones: el país del cliente y su condición de nuevo o recurrente. El criterio de priorización es la contribución absoluta en libras y no la variación porcentual: un mercado pequeño puede caer 78 % y pesar mucho menos que un mercado grande que cae 1 %.


In [ ]:
# Segmentación explicativa por país. Interesan dos criterios distintos: qué
# mercado explica más libras de la variación (contribución absoluta) y qué
# mercado se deterioró más respecto de sí mismo (variación porcentual). No
# siempre son el mismo, y confundirlos lleva a priorizar mal.
por_pais = (
    comparables
    .group_by(["pais", "periodo"])
    .agg(pl.col("ingreso").sum())
    .pivot(values="ingreso", index="pais", on="periodo", aggregate_function="first")
    .fill_null(0.0)
    .with_columns((pl.col("P1") - pl.col("P0")).alias("variacion"))
    .with_columns(
        pl.when(pl.col("P0") > 0)
          .then(100 * pl.col("variacion") / pl.col("P0"))
          .otherwise(None)
          .alias("variacion_pct")
    )
    .sort("variacion")
)

print("Mayores caídas en libras:")
print(por_pais.head(6))
print("\nMayores aumentos en libras:")
print(por_pais.tail(6))

Mayores caídas en libras:
shape: (6, 5)
┌──────────────────────┬────────────────────┬────────────────────┬─────────────────────┬─────────────────────┐
│ pais                 ┆ P1                 ┆ P0                 ┆ variacion           ┆ variacion_pct       │
│ ---                  ┆ ---                ┆ ---                ┆ ---                 ┆ ---                 │
│ str                  ┆ f64                ┆ f64                ┆ f64                 ┆ f64                 │
╞══════════════════════╪════════════════════╪════════════════════╪═════════════════════╪═════════════════════╡
│ EIRE                 ┆ 263709.9699999999  ┆ 352563.3000000046  ┆ -88853.33000000467  ┆ -25.2020927873104   │
│ Denmark              ┆ 18060.439999999988 ┆ 49211.35000000002  ┆ -31150.910000000033 ┆ -63.30025492086687  │
│ Sweden               ┆ 36590.82999999997  ┆ 49490.310000000056 ┆ -12899.480000000083 ┆ -26.064657909801067 │
│ Greece               ┆ 3879.53            ┆ 14285.670000000011 ┆ -1040

In [ ]:
# Segunda dimensión: clientes nuevos frente a recurrentes. Un cliente es
# recurrente en P1 si ya había comprado en P0.
clientes_p0 = (comparables.filter(pl.col("periodo") == "P0")
               ["cliente_id"].unique().to_list())

tipos = (
    comparables
    .filter((pl.col("periodo") == "P1") & pl.col("cliente_id").is_not_null())
    .with_columns(
        pl.when(pl.col("cliente_id").is_in(clientes_p0))
          .then(pl.lit("recurrente"))
          .otherwise(pl.lit("nuevo"))
          .alias("tipo")
    )
    .group_by("tipo")
    .agg(pl.col("ingreso").sum().round(2).alias("ingresos"),
         pl.col("cliente_id").n_unique().alias("clientes"))
    .with_columns((pl.col("ingresos") / pl.col("clientes"))
                  .round(2).alias("ingreso_por_cliente"))
)
print(tipos)

shape: (2, 4)
┌────────────┬────────────┬──────────┬─────────────────────┐
│ tipo       ┆ ingresos   ┆ clientes ┆ ingreso_por_cliente │
│ ---        ┆ ---        ┆ ---      ┆ ---                 │
│ str        ┆ f64        ┆ u32      ┆ f64                 │
╞════════════╪════════════╪══════════╪═════════════════════╡
│ nuevo      ┆ 1448212.66 ┆ 1585     ┆ 913.7               │
│ recurrente ┆ 6776786.9  ┆ 2708     ┆ 2502.51             │
└────────────┴────────────┴──────────┴─────────────────────┘


In [ ]:
# La misma pregunta resuelta en SQL. DuckDB consulta directamente el DataFrame
# de Polars, sin necesidad de cargarlo en una base de datos: en un entorno de
# trabajo real, buena parte del equipo leerá SQL antes que Polars.
consulta = duckdb.sql("""
    SELECT  pais,
            SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) AS p0,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE 0 END) AS p1,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE -ingreso END) AS variacion,
            COUNT(DISTINCT cliente_id) AS clientes
    FROM    comparables
    GROUP BY pais
    HAVING  SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) > 20000
    ORDER BY variacion
""").pl()

print(consulta)

shape: (13, 5)
┌─────────────────┬────────────────────┬────────────────────┬─────────────────────┬──────────┐
│ pais            ┆ p0                 ┆ p1                 ┆ variacion           ┆ clientes │
│ ---             ┆ ---                ┆ ---                ┆ ---                 ┆ ---      │
│ str             ┆ f64                ┆ f64                ┆ f64                 ┆ i64      │
╞═════════════════╪════════════════════╪════════════════════╪═════════════════════╪══════════╡
│ EIRE            ┆ 352563.29999999964 ┆ 263709.9699999988  ┆ -88853.33000000028  ┆ 3        │
│ Denmark         ┆ 49211.35000000003  ┆ 18060.439999999995 ┆ -31150.910000000018 ┆ 11       │
│ Sweden          ┆ 49490.31000000002  ┆ 36590.83           ┆ -12899.479999999981 ┆ 19       │
│ Channel Islands ┆ 24032.78999999998  ┆ 19799.139999999992 ┆ -4233.65            ┆ 13       │
│ Portugal        ┆ 20387.160000000018 ┆ 24534.92           ┆ 4147.760000000005   ┆ 23       │
│ Netherlands     ┆ 265884.06999999

In [ ]:
# Cuadro final de drivers, ordenado por contribución absoluta. Este es el
# insumo de la decisión: el driver dominante es el de mayor peso en libras,
# no el que resulte más llamativo en porcentaje.
drivers = pl.DataFrame({
    "driver": ["Volumen del catálogo común", "Mezcla del catálogo común",
               "Precio del catálogo común", "Productos nuevos",
               "Productos discontinuados"],
    "contribucion": [efecto_volumen, efecto_mezcla, efecto_precio,
                     nuevos["r"].sum(), -discontinuados["r"].sum()],
}).with_columns(
    pl.col("contribucion").round(2),
    pl.col("contribucion").abs().alias("magnitud"),
).with_columns(
    (100 * pl.col("magnitud") / pl.col("magnitud").sum())
        .round(1).alias("peso_relativo"),
).sort("magnitud", descending=True).drop("magnitud")

print(drivers)
print(f"\nVariación neta de ingresos: £ {variacion_total:,.2f}")
print("Los cinco drivers suman esa variación neta, pero cada uno la supera")
print("ampliamente en magnitud: se compensan entre sí. Ese es el hallazgo.")

shape: (5, 3)
┌────────────────────────────┬──────────────┬───────────────┐
│ driver                     ┆ contribucion ┆ peso_relativo │
│ ---                        ┆ ---          ┆ ---           │
│ str                        ┆ f64          ┆ f64           │
╞════════════════════════════╪══════════════╪═══════════════╡
│ Productos nuevos           ┆ 2046903.92   ┆ 40.8          │
│ Volumen del catálogo común ┆ -1568093.05  ┆ 31.3          │
│ Productos discontinuados   ┆ -704357.16   ┆ 14.1          │
│ Mezcla del catálogo común  ┆ 577550.82    ┆ 11.5          │
│ Precio del catálogo común  ┆ -115918.44   ┆ 2.3           │
└────────────────────────────┴──────────────┴───────────────┘

Variación neta de ingresos: £ 236,086.10
Los cinco drivers suman esa variación neta, pero cada uno la supera
ampliamente en magnitud: se compensan entre sí. Ese es el hallazgo.


**Pregunta 7 — Priorización**

a) Identifique el país con mayor caída en libras y el país con mayor caída porcentual. ¿Son el mismo? b) Si la gerencia solo pudiera atender un mercado, ¿cuál recomendaría y por qué? c) ¿Qué información adicional, no contenida en este conjunto de datos, necesitaría para sostener una afirmación causal?

*Respuesta:* a) EIRE (Irlanda) es el país con la mayor caída en libras: £-88,853.33 (-25.20 %). Pero si se mira la caída porcentual entre los mercados con una base de ingresos relevante, el que más cae es Emiratos Árabes Unidos, con -77.82 % (£-6,543.67); no coinciden, son mercados distintos.

*Respuesta:* b) Priorizaría EIRE. Su caída porcentual no es la más dramática, pero concentra la mayor pérdida de ingresos en términos absolutos entre todos los mercados internacionales, así que recuperarlo pesa más en el resultado que recuperar otros mercados con caídas porcentuales mayores pero de escala mucho más chica.

*Respuesta:* c) Haría falta saber, por ejemplo, qué está haciendo la competencia, si hubo cambios en marketing o en el gasto publicitario por mercado, si hubo quiebres de stock, y las condiciones macroeconómicas o cambiarias de cada país. En general, cualquier variable externa al registro transaccional que pueda haber influido en la decisión de compra de esos clientes.

### RETO FINAL — Del diagnóstico a la decisión

*5 minutos*

Complete la última celda del notebook con el driver dominante, la cifra que lo sustenta y la limitación de su análisis. Las tres cosas se exigen juntas: un driver sin cifra es una opinión, y una cifra sin limitación es una conclusión sobrevendida.


---

## Reto de aplicación y retroalimentación

En esta sección se aplicarán los procedimientos desarrollados durante la sesión a nuevas situaciones de análisis. Cada ejercicio requiere modificar, completar o construir código a partir de las tablas ya procesadas. Posteriormente, los resultados obtenidos deberán interpretarse brevemente desde una perspectiva empresarial. El propósito es comprobar la comprensión de las técnicas utilizadas y fortalecer la capacidad de adaptar el análisis ante nuevas preguntas de negocio.

**Indicaciones generales.** Los ejercicios operan sobre las tablas `comparables`, `por_pais`, `retencion`, `matriz` y `drivers`, ya construidas durante la sesión. Los datos proceden íntegramente de la fuente declarada en la ficha de trazabilidad; no corresponde generar ni sustituir valores en ningún caso. Cada respuesta escrita no debe exceder cuatro líneas.

**Duración en sesión:** 25 minutos. Los ejercicios que no concluyan se completan como avance del entregable.


### Ejercicio 1 — Identificación y ponderación del efecto dominante

El puente construido durante la sesión descompuso la variación del catálogo común en tres efectos: volumen, mezcla y precio. El cuadro de drivers los presentó ordenados, pero la lectura gerencial exige una precisión adicional: establecer cuál de ellos concentra la mayor magnitud y qué proporción representa respecto de la variación neta observada.

Esa proporción resulta decisiva. Cuando los efectos se compensan entre sí, cada uno puede superar ampliamente en magnitud a la variación neta, situación que invalida la lectura habitual según la cual el efecto dominante explica la mayor parte del cambio.

**Se solicita:**

1. Determinar mediante código cuál de los tres efectos presenta el mayor valor absoluto.
2. Expresar cada efecto como porcentaje de la variación neta de ingresos.
3. Comprobar si la suma de los valores absolutos de los tres efectos supera la variación neta.
4. Registrar el resultado en una tabla ordenada por magnitud.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 1
# ---------------------------------------------------------------------------
efectos = {
    "volumen": efecto_volumen,
    "mezcla":  efecto_mezcla,
    "precio":  efecto_precio,
}

dominante = max(efectos, key=lambda k: abs(efectos[k]))
print(f"Efecto dominante (mayor valor absoluto): {dominante} -> £ {efectos[dominante]:,.2f}\n")

tabla_ej1 = (
    pl.DataFrame({"efecto": list(efectos.keys()), "valor": list(efectos.values())})
    .with_columns(
        pl.col("valor").abs().alias("magnitud"),
        (100 * pl.col("valor") / variacion_total).round(1).alias("pct_de_variacion_neta"),
    )
    .sort("magnitud", descending=True)
    .drop("magnitud")
)
print(tabla_ej1)

suma_abs = sum(abs(v) for v in efectos.values())
print(f"\nSuma de valores absolutos de los tres efectos : £ {suma_abs:,.2f}")
print(f"Variación neta de ingresos                    : £ {variacion_total:,.2f}")
print("¿La suma de los valores absolutos supera la variación neta?",
      "Sí" if suma_abs > abs(variacion_total) else "No")


Efecto dominante (mayor valor absoluto): volumen -> £ -1,568,093.05

shape: (3, 3)
┌─────────┬─────────────────────┬───────────────────────┐
│ efecto  ┆ valor               ┆ pct_de_variacion_neta │
│ ---     ┆ ---                 ┆ ---                   │
│ str     ┆ f64                 ┆ f64                   │
╞═════════╪═════════════════════╪═══════════════════════╡
│ volumen ┆ -1568093.0468426752 ┆ -664.2                │
│ mezcla  ┆ 577550.82420766     ┆ 244.6                 │
│ precio  ┆ -115918.43736498346 ┆ -49.1                 │
└─────────┴─────────────────────┴───────────────────────┘

Suma de valores absolutos de los tres efectos : £ 2,261,562.31
Variación neta de ingresos                    : £ 236,086.10
¿La suma de los valores absolutos supera la variación neta? Sí


**Pregunta de cierre.** ¿El efecto dominante explica por sí solo la variación neta observada? Justifique con las cifras obtenidas.

**Respuesta (máximo cuatro líneas):** No. De los tres, el efecto volumen es el que domina (£-1,568,093.05), pero eso equivale a -664.2 % de la variación neta (£236,086.10), es decir, la supera ampliamente y en sentido contrario. Sumados en valor absoluto, los tres efectos llegan a £2,261,562.31, casi diez veces la variación neta observada, lo que confirma que se compensan entre sí y que ninguno explica el resultado agregado por sí solo.

---


### Ejercicio 2 — Aplicación de la descomposición a un mercado específico

La segmentación por país identificó los mercados de mayor caída en libras. Esa información establece dónde se produjo el deterioro, pero no por qué se produjo: un mercado puede retroceder porque compra menos unidades, porque desplaza su compra hacia productos de menor valor o porque obtiene precios inferiores.

Aplicar el puente precio-volumen-mezcla al interior de un mercado específico permite responder esa pregunta y constituye el procedimiento habitual cuando la gerencia solicita explicar el resultado de una plaza determinada.

**Se solicita:**

1. Identificar el país con la mayor caída en libras, excluyendo el Reino Unido por su tamaño relativo.
2. Delimitar las transacciones de ese país en ambos periodos.
3. Replicar sobre ese subconjunto la descomposición en efectos de volumen, mezcla y precio, empleando el catálogo común.
4. Comprobar que la suma de los efectos reconcilia con la variación del país.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 2
# ---------------------------------------------------------------------------
# 1. País con mayor caída en libras, excluyendo Reino Unido.
pais_objetivo = (
    por_pais.filter(pl.col("pais") != "United Kingdom")
    .sort("variacion")
    .head(1)
    .row(0, named=True)["pais"]
)
print("País objetivo:", pais_objetivo,
      f"(variación £ {por_pais.filter(pl.col('pais')==pais_objetivo)['variacion'][0]:,.2f})")

# 2. Transacciones de ese país en ambos periodos.
del_pais = comparables.filter(pl.col("pais") == pais_objetivo)

# 3. Descomposición precio-volumen-mezcla sobre el catálogo común de ese país.
pp_pais = (
    del_pais.group_by(["periodo", "codigo"])
    .agg(pl.col("cantidad").sum().alias("q"), pl.col("ingreso").sum().alias("r"))
    .with_columns((pl.col("r") / pl.col("q")).alias("p"))
)
p0_pais = pp_pais.filter(pl.col("periodo") == "P0").drop("periodo")
p1_pais = pp_pais.filter(pl.col("periodo") == "P1").drop("periodo")

comun_pais  = p0_pais.join(p1_pais, on="codigo", suffix="_1")
disc_pais   = p0_pais.join(p1_pais, on="codigo", how="anti")
nuevos_pais = p1_pais.join(p0_pais, on="codigo", how="anti")

Q0p, Q1p = comun_pais["q"].sum(), comun_pais["q_1"].sum()
cp = comun_pais.with_columns((pl.col("q") / Q0p).alias("s0"), (pl.col("q_1") / Q1p).alias("s1"))

ev_pais = (Q1p - Q0p) * (cp["s0"] * cp["p"]).sum()
em_pais = Q1p * ((cp["s1"] - cp["s0"]) * cp["p"]).sum()
ep_pais = Q1p * (cp["s1"] * (cp["p_1"] - cp["p"])).sum()

var_comun_pais = cp["r_1"].sum() - cp["r"].sum()
var_total_pais = p1_pais["r"].sum() - p0_pais["r"].sum()

print(f"\nEfecto volumen : £ {ev_pais:,.2f}")
print(f"Efecto mezcla  : £ {em_pais:,.2f}")
print(f"Efecto precio  : £ {ep_pais:,.2f}")
print(f"Suma           : £ {ev_pais + em_pais + ep_pais:,.2f}")

# 4. Comprobación de reconciliación.
assert abs(var_comun_pais - (ev_pais + em_pais + ep_pais)) < 0.01, "No reconcilia"
print(f"\nVariación catálogo común  : £ {var_comun_pais:,.2f}  (reconcilia con la suma de efectos)")
print(f"Productos nuevos          : £ {nuevos_pais['r'].sum():,.2f}")
print(f"Productos discontinuados  : £ {-disc_pais['r'].sum():,.2f}")
print(f"Variación total del país  : £ {var_total_pais:,.2f}")


País objetivo: EIRE (variación £ -88,853.33)

Efecto volumen : £ -94,481.17
Efecto mezcla  : £ 7,550.65
Efecto precio  : £ -8,785.37
Suma           : £ -95,715.89

Variación catálogo común  : £ -95,715.89  (reconcilia con la suma de efectos)
Productos nuevos          : £ 81,175.57
Productos discontinuados  : £ -74,313.01
Variación total del país  : £ -88,853.33


**Pregunta de cierre.** ¿Qué efecto explica la caída de ese mercado y en qué se diferencia del patrón observado en el agregado?

**Respuesta (máximo cuatro líneas):** En EIRE, el efecto volumen (£-94,481.17) domina la caída con amplio margen; el efecto mezcla también es positivo, pero mucho más chico (£7,550.65) que el £577,550.82 del agregado. Esto contrasta con lo que pasa a nivel de toda la empresa, donde volumen, mezcla y precio se compensan de forma muy fuerte entre sí (cada uno muy por encima de la variación neta total). En EIRE, en cambio, los tres efectos tienen una magnitud parecida a la variación observada del país: simplemente se vendieron menos unidades, sin que un cambio de mezcla lo compensara.

---


### Ejercicio 3 — Comparación de la retención entre cohortes

La matriz de retención presenta la evolución de cada cohorte a lo largo de su antigüedad. Su lectura completa resulta exigente, y por ello la práctica habitual consiste en fijar un punto de corte —por ejemplo, el sexto mes— y comparar allí todas las cohortes que ya lo alcanzaron.

Ese corte responde a una pregunta directa: si la calidad de los clientes captados mejora o se deteriora con el paso del tiempo. Corresponde excluir del análisis las cohortes que aún no cumplieron el plazo, dado que sus celdas vacías no representan abandono sino ausencia de información.

**Se solicita:**

1. Extraer la tasa de retención correspondiente al sexto mes de antigüedad de cada cohorte.
2. Excluir de manera explícita las cohortes que no alcanzaron esa antigüedad.
3. Ordenar las cohortes cronológicamente y comparar sus tasas.
4. Determinar si existe una tendencia de mejora o de deterioro entre las cohortes más antiguas y las más recientes.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 3
# ---------------------------------------------------------------------------
# 1 y 2. Retención al mes 6, excluyendo cohortes que aún no lo alcanzaron.
tope_cohorte = retencion.group_by("cohorte").agg(pl.col("antiguedad").max().alias("tope"))

retencion_mes6 = (
    retencion
    .filter(pl.col("antiguedad") == 6)
    .join(tope_cohorte, on="cohorte")
    .filter(pl.col("tope") >= 6)
    # 3. orden cronológico
    .sort("cohorte")
    .select("cohorte", pl.col("tasa").round(2))
)
print(f"Cohortes con al menos 6 meses de antigüedad: {retencion_mes6.height}\n")
print(retencion_mes6)

# 4. Tendencia: promedio de la primera mitad cronológica frente a la segunda.
n = retencion_mes6.height
primera_mitad = retencion_mes6.head(n // 2)["tasa"].mean()
segunda_mitad = retencion_mes6.tail(n // 2)["tasa"].mean()
print(f"\nPromedio retención mes 6 — primeras {n//2} cohortes  : {primera_mitad:.2f} %")
print(f"Promedio retención mes 6 — últimas {n//2} cohortes    : {segunda_mitad:.2f} %")
print("Tendencia:", "deterioro" if segunda_mitad < primera_mitad else "mejora")


Cohortes con al menos 6 meses de antigüedad: 19

shape: (19, 2)
┌────────────┬───────┐
│ cohorte    ┆ tasa  │
│ ---        ┆ ---   │
│ date       ┆ f64   │
╞════════════╪═══════╡
│ 2009-12-01 ┆ 37.64 │
│ 2010-01-01 ┆ 26.9  │
│ 2010-02-01 ┆ 19.2  │
│ 2010-03-01 ┆ 24.72 │
│ 2010-04-01 ┆ 27.55 │
│ 2010-05-01 ┆ 21.18 │
│ 2010-06-01 ┆ 12.73 │
│ 2010-07-01 ┆ 11.35 │
│ 2010-08-01 ┆ 9.82  │
│ 2010-09-01 ┆ 13.81 │
│ 2010-10-01 ┆ 13.07 │
│ 2010-11-01 ┆ 12.88 │
│ 2010-12-01 ┆ 5.26  │
│ 2011-01-01 ┆ 15.28 │
│ 2011-02-01 ┆ 16    │
│ 2011-03-01 ┆ 20.67 │
│ 2011-04-01 ┆ 17.92 │
│ 2011-05-01 ┆ 26.13 │
│ 2011-06-01 ┆ 8.33  │
└────────────┴───────┘

Promedio retención mes 6 — primeras 9 cohortes  : 21.23 %
Promedio retención mes 6 — últimas 9 cohortes    : 15.06 %
Tendencia: deterioro


**Pregunta de cierre.** ¿Las cohortes recientes retienen mejor o peor que las antiguas? Indique qué implicancia tiene ese resultado para la captación de clientes.

**Respuesta (máximo cuatro líneas):** Retienen peor. El promedio de retención al mes 6, entre las 19 cohortes con al menos esa antigüedad, baja de 21.23 % en las primeras a 15.06 % en las últimas, con un contraste marcado entre la cohorte de diciembre de 2009 (37.64 %) y la de junio de 2011 (8.33 %). Esto implica que los clientes nuevos ya no compensan, en calidad, a los de las primeras cohortes: si se sigue creciendo solo en número sin revertir esta caída en retención, la base de ingresos recurrentes se debilita en el mediano plazo.

---


### Ejercicio 4 — Consulta de mercados prioritarios con condiciones múltiples en DuckDB

Durante la sesión se empleó DuckDB para agregar los ingresos por país en ambos periodos. Una consulta orientada a la decisión incorpora, además, las restricciones que la gerencia impone antes de asignar recursos.

Un mercado en retroceso no constituye por sí mismo una prioridad: puede tratarse de una plaza pequeña cuya caída carece de relevancia económica, o de un mercado atendido por muy pocos clientes, en cuyo caso el retroceso obedece a la pérdida de una cuenta y no a un deterioro del mercado.

**Se solicita:**

1. Construir sobre la tabla `comparables` una consulta SQL que incluya `SELECT`, `GROUP BY`, `HAVING` y `ORDER BY`.
2. Calcular por país los ingresos de cada periodo, la variación absoluta y la cantidad de clientes distintos.
3. Conservar únicamente los mercados con variación negativa, ingresos del primer periodo superiores a un umbral declarado explícitamente y más de una cantidad mínima de clientes.
4. Ordenar el resultado por variación de manera ascendente.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 4
# ---------------------------------------------------------------------------
UMBRAL_INGRESOS_P0 = 20_000   # libras: se excluyen mercados demasiado pequeños
MINIMO_CLIENTES    = 10       # se excluyen mercados sostenidos por muy pocas cuentas

mercados_prioritarios = duckdb.sql(f"""
    SELECT  pais,
            SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) AS ingresos_p0,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE 0 END) AS ingresos_p1,
            SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE -ingreso END) AS variacion,
            COUNT(DISTINCT cliente_id) AS clientes
    FROM    comparables
    GROUP BY pais
    HAVING  SUM(CASE WHEN periodo = 'P1' THEN ingreso ELSE -ingreso END) < 0
        AND SUM(CASE WHEN periodo = 'P0' THEN ingreso ELSE 0 END) > {UMBRAL_INGRESOS_P0}
        AND COUNT(DISTINCT cliente_id) > {MINIMO_CLIENTES}
    ORDER BY variacion ASC
""").pl()

print(f"Umbral de ingresos P0 : £ {UMBRAL_INGRESOS_P0:,}")
print(f"Mínimo de clientes    : {MINIMO_CLIENTES}\n")
print(mercados_prioritarios)


Umbral de ingresos P0 : £ 20,000
Mínimo de clientes    : 10

shape: (3, 5)
┌─────────────────┬───────────────────┬────────────────────┬─────────────────────┬──────────┐
│ pais            ┆ ingresos_p0       ┆ ingresos_p1        ┆ variacion           ┆ clientes │
│ ---             ┆ ---               ┆ ---                ┆ ---                 ┆ ---      │
│ str             ┆ f64               ┆ f64                ┆ f64                 ┆ i64      │
╞═════════════════╪═══════════════════╪════════════════════╪═════════════════════╪══════════╡
│ Denmark         ┆ 49211.35000000002 ┆ 18060.439999999995 ┆ -31150.910000000025 ┆ 11       │
│ Sweden          ┆ 49490.31000000003 ┆ 36590.82999999999  ┆ -12899.47999999999  ┆ 19       │
│ Channel Islands ┆ 24032.78999999998 ┆ 19799.139999999992 ┆ -4233.65            ┆ 13       │
└─────────────────┴───────────────────┴────────────────────┴─────────────────────┴──────────┘


**Pregunta de cierre.** ¿Cuántos mercados satisfacen los tres criterios y cuál encabeza el ordenamiento? ¿Coincide con el de mayor caída porcentual?

**Respuesta (máximo cuatro líneas):** Con un umbral de £20,000 en ingresos P0 y un mínimo de 10 clientes distintos, quedan tres mercados que cumplen los tres criterios: Dinamarca, Suecia y Channel Islands. Dinamarca encabeza con £-31,150.91, y dentro de este grupo también tiene la mayor caída porcentual (-63.30 %). Eso sí, no es la mayor caída porcentual de todo el dataset, título que corresponde a Emiratos Árabes Unidos (-77.82 %), un mercado muy pequeño que queda fuera de este filtro por no alcanzar el umbral de tamaño.

---


### Ejercicio 5 — Representación conjunta de la magnitud y la intensidad del deterioro

La segmentación por país produjo dos ordenamientos distintos: uno por variación en libras y otro por variación porcentual. Ambos responden preguntas diferentes y rara vez coinciden, de modo que examinarlos por separado induce a error en la priorización.

El gráfico de dispersión resuelve esa dificultad al presentar simultáneamente la magnitud económica del deterioro y su intensidad relativa. Los mercados situados en el cuadrante de caída elevada en ambas dimensiones constituyen la prioridad; los de gran caída absoluta pero deterioro relativo moderado responden a un problema de escala y no de mercado.

**Se solicita:**

1. Construir con Plotly un gráfico de dispersión con la variación en libras en el eje horizontal y la variación porcentual en el eje vertical.
2. Representar los ingresos del primer periodo mediante el tamaño del marcador.
3. Identificar cada punto con el nombre del país y excluir el Reino Unido, cuya escala impide la lectura del resto.
4. Rotular los ejes y titular el gráfico de modo que resulte interpretable sin recurrir al código.

**Tiempo estimado:** 5 minutos.


In [ ]:
# ---------------------------------------------------------------------------
# CELDA DE TRABAJO — Ejercicio 5
# ---------------------------------------------------------------------------
datos_ej5 = (
    por_pais
    .filter(pl.col("pais") != "United Kingdom")
    .filter(pl.col("P0") > 0)   # variación porcentual solo tiene sentido con base positiva
)

figura = px.scatter(
    datos_ej5.to_pandas(),
    x="variacion", y="variacion_pct",
    size="P0", text="pais",
    labels={"variacion": "Variación de ingresos (£)",
            "variacion_pct": "Variación de ingresos (%)",
            "P0": "Ingresos P0 (£)"},
)
figura.update_traces(textposition="top center")
figura.update_layout(
    title="Magnitud absoluta vs. intensidad relativa de la caída por país (excluye Reino Unido)",
    height=560,
)
figura.show()

peor_en_ambas = (
    datos_ej5.filter(pl.col("variacion") < 0)
    .sort(["variacion", "variacion_pct"])
    .head(5)
)
print(peor_en_ambas.select("pais", "P0", "variacion", "variacion_pct"))


shape: (5, 4)
┌──────────────────────┬────────────────────┬─────────────────────┬─────────────────────┐
│ pais                 ┆ P0                 ┆ variacion           ┆ variacion_pct       │
│ ---                  ┆ ---                ┆ ---                 ┆ ---                 │
│ str                  ┆ f64                ┆ f64                 ┆ f64                 │
╞══════════════════════╪════════════════════╪═════════════════════╪═════════════════════╡
│ EIRE                 ┆ 352563.3000000046  ┆ -88853.33000000467  ┆ -25.2020927873104   │
│ Denmark              ┆ 49211.35000000002  ┆ -31150.910000000033 ┆ -63.30025492086687  │
│ Sweden               ┆ 49490.310000000056 ┆ -12899.480000000083 ┆ -26.064657909801067 │
│ Greece               ┆ 14285.670000000011 ┆ -10406.14000000001  ┆ -72.84320581393804  │
│ United Arab Emirates ┆ 8408.449999999988  ┆ -6543.669999999987  ┆ -77.82254755632722  │
└──────────────────────┴────────────────────┴─────────────────────┴───────────────────

**Pregunta de cierre.** ¿Qué mercado presenta simultáneamente caída elevada en libras y en porcentaje? ¿Por qué constituye una prioridad distinta de un mercado con gran caída absoluta pero deterioro relativo moderado?

**Respuesta (máximo cuatro líneas):** Dinamarca cae fuerte en las dos dimensiones a la vez: £-31,150.91 (la segunda mayor caída absoluta, después de EIRE) y -63.30 % (de las más altas entre los mercados con base relevante). Esto la diferencia de EIRE, que tiene la mayor caída absoluta (£-88,853.33) pero un deterioro relativo moderado (-25.20 %): en EIRE el problema es de escala —un mercado grande que cae poco en proporción igual pesa mucho en libras—, mientras que en Dinamarca el problema es del mercado en sí, porque perdió más de la mitad de su negocio relativo.

---


### Del diagnóstico a la decisión

Corresponde cerrar el laboratorio con una decisión sustentada. Un diagnóstico que no reconoce sus propios límites no constituye un diagnóstico riguroso.

**Pregunta de decisión.** A partir del cuadro de drivers y de los resultados de los cinco ejercicios, indique:

1. El driver de mayor contribución absoluta y la cifra en libras que lo sustenta.
2. La acción empresarial que recomienda en consecuencia.
3. La limitación del análisis: qué **no** demuestra este trabajo, incluso habiendo reconciliado todos los efectos.

**Respuesta:** 1) El driver con mayor contribución absoluta es "productos nuevos", con £2,046,903.92 (40.8 % del peso relativo dentro del cuadro de drivers). Muy cerca le sigue el efecto volumen del catálogo común, con £-1,568,093.05 (31.3 %), pero en sentido contrario. Ambos superan por mucho la variación neta observada (£236,086.10), lo que confirma que el crecimiento de ingresos reportado nace de fuerzas que se compensan entre sí, y no de una mejora generalizada del negocio.

2) Como primera acción, invertiría en revertir la caída de volumen del catálogo existente, que es el problema de mayor peso entre los tres efectos PVM, enfocándome en EIRE, donde se concentra la mayor caída de ingresos en libras (£-88,853.33). Como segunda acción, evaluaría si el aporte de £2,046,903.92 de los productos nuevos es sostenible en el tiempo o depende de artículos de rotación puntual, antes de apostar la estrategia comercial a que se mantenga.

3) Lo que este análisis no demuestra es causalidad. Explica matemáticamente qué componentes mueven la variación de ingresos y dónde se concentran, pero no responde por qué los clientes compraron menos unidades del catálogo común ni por qué las cohortes recientes retienen menos. Para poder afirmar algo causal se necesitarían datos que este conjunto no tiene: competencia, marketing, macroeconomía, satisfacción del cliente.

---


**Ticket de salida**

Responda antes de retirarse de la sesión.

**Ticket 1**

¿Por qué pueden subir los ingresos mientras caen las unidades? Explique usando los conceptos de efecto volumen, efecto mezcla y efecto precio.

*Respuesta:* El ingreso no depende únicamente de cuántas unidades se venden, sino también de qué se vende y a qué precio efectivo. Aquí las unidades cayeron 6.72 % y el efecto volumen fue muy negativo (£-1,568,093.05), pero eso se compensó de sobra con el ingreso de los productos nuevos (£2,046,903.92) y con un efecto mezcla positivo (£577,550.82): se vendió menos cantidad, pero lo vendido tuvo, en conjunto, mayor valor unitario, y además el catálogo sumó artículos que aportaron ingresos extra.

*Respuesta:* De ahí que el ingreso pueda subir 2.51 % justo cuando las unidades caen 6.72 %: son dos medidas de cosas distintas, y no hay razón para que se muevan en la misma dirección ni con la misma magnitud.

**Ticket 2**

¿Qué variable debe mantenerse constante para que la comparación entre dos cohortes sea válida?

*Respuesta:* Lo que debe mantenerse fijo es la antigüedad —los meses transcurridos desde la primera compra de cada cohorte—, no el mes del calendario. Dos cohortes solo pueden compararse en el mismo punto de su ciclo de vida (por ejemplo, ambas en su mes 6); compararlas en el mismo mes calendario no tiene sentido si nacieron en fechas distintas.

**Ticket 3**

Indique el driver dominante del caso, una cifra que lo sustente y una limitación de la conclusión.

*Respuesta:* El driver dominante es "productos nuevos", con £2,046,903.92 de contribución (40.8 % del peso relativo del cuadro de drivers); en segundo lugar aparece el efecto volumen del catálogo común, con £-1,568,093.05 (31.3 %).

*Respuesta:* Las dos cifras superan por mucho, y en sentidos opuestos, la variación neta de ingresos observada (£236,086.10), lo que deja claro que ese crecimiento neto viene de fuerzas que se compensan entre sí y no de una mejora generalizada del negocio.

*Respuesta:* La limitación es que el análisis no prueba causalidad: identifica matemáticamente dónde se concentra el cambio, pero no explica sus causas (competencia, marketing, macroeconomía), y esa información no está en este conjunto de datos.

---

## Actividad 3 — Informe para la gerencia

Responda en términos de negocio y cite cifras obtenidas en el notebook. Toda afirmación sin cifra se considera no sustentada.

**A.**

La gerencia sostiene que el año fue mejor porque los ingresos crecieron. ¿Comparte esa lectura? Fundamente con la descomposición obtenida.

*Respuesta:* No comparto esa lectura tal cual. Es verdad que los ingresos subieron 2.51 % (de £9,396,642.30 a £9,632,728.40), pero en el mismo periodo las unidades vendidas bajaron 6.72 %, el número de facturas bajó 3.98 % y el catálogo activo perdió 316 productos distintos (-7.49 %).

*Respuesta:* Al descomponer el resultado se ve que ese crecimiento no viene de vender más ni de vender mejor el mismo catálogo: el efecto volumen del catálogo común (£-1,568,093.05) y el efecto precio (£-115,918.44) son ambos negativos. Lo que sostuvo el número final fue la incorporación de productos nuevos (£2,046,903.92) más un cambio favorable en la mezcla del catálogo ya existente (£577,550.82).

*Respuesta:* Entonces, el crecimiento de ingresos es real en libras, pero convive con una caída del volumen de actividad; decir que "el año fue mejor" solo por el ingreso es una lectura incompleta.

**B.**

¿Cuál es el driver dominante de la variación y qué proporción del movimiento total explica?

*Respuesta:* El driver con mayor contribución absoluta es "productos nuevos", con £2,046,903.92, el 40.8 % del peso relativo del cuadro de los cinco drivers.

*Respuesta:* Detrás está el efecto volumen del catálogo común, £-1,568,093.05 (31.3 % del peso relativo), pero apuntando en la dirección contraria.

*Respuesta:* Ninguno de los dos alcanza por sí solo a "explicar" la variación neta de £236,086.10: ambos la superan varias veces en magnitud, y el resultado final es en realidad la diferencia entre fuerzas que se compensan, no el efecto de un solo driver dominante.

**C.**

El catálogo perdió productos y ganó otros. ¿Qué decisión comercial concreta recomendaría respecto del catálogo y con qué cifra la sustenta?

*Respuesta:* Antes de dar por hecho que los productos nuevos son una base sostenible de crecimiento, recomendaría revisar el catálogo a fondo: de 4,220 productos activos en P0 se pasó a 3,904 en P1 (990 se discontinuaron, y representaban £704,357.16 en P0; 674 son nuevos y aportaron £2,046,903.92 en P1).

*Respuesta:* Como los productos nuevos son el driver de mayor peso (40.8 %), habría que mirar su rotación y su margen para saber si vale la pena invertir en sostener ese catálogo emergente, o si su aporte fue algo puntual que no se va a repetir.

*Respuesta:* En paralelo, investigaría por qué se dejaron de vender productos que sí generaban £704,357.16 en ingresos, para descartar que haya sido por quiebres de stock evitables y no por una decisión comercial pensada.

**D.**

A partir del análisis de cohortes y de la segmentación, señale un segmento de clientes que priorizaría y explique por qué ese y no otro.

*Respuesta:* Priorizaría a los clientes recurrentes. En P1 generaron £6,776,786.90, frente a £1,448,212.66 de los clientes nuevos, y su ingreso promedio por cliente (£2,502.51) es casi 2.7 veces el de un cliente nuevo (£913.70).

*Respuesta:* A esto se suma que la matriz de cohortes muestra una retención al mes 6 cada vez más baja (de 37.64 % en diciembre de 2009 a 8.33 % en junio de 2011), así que fortalecer la relación con los clientes recurrentes que ya tiene la empresa da un efecto más predecible sobre el ingreso que seguir captando clientes nuevos con una retención que va cayendo.

*Respuesta:* Dentro de ese segmento, geográficamente priorizaría EIRE, porque es el país que concentra la mayor caída de ingresos en libras (£-88,853.33).

**E.**

Enuncie dos afirmaciones que este análisis NO permite sostener, y qué evidencia adicional se necesitaría para sostenerlas.

*Respuesta:* 1) No se puede sostener que "la calidad del servicio empeoró", aunque las cohortes recientes retengan menos que las antiguas (15.06 % frente a 21.23 % en promedio al mes 6). Para poder decir eso harían falta datos de satisfacción del cliente, quejas o incidencias de servicio, que este conjunto de datos no tiene.

*Respuesta:* 2) Tampoco se puede sostener que "la empresa está perdiendo mercado frente a la competencia" solo porque las unidades vendidas cayeron 6.72 %. Para eso se necesitaría saber cómo evolucionó el mercado total (no solo esta empresa) y qué pasó con las ventas de competidores directos en el mismo periodo.

*Respuesta:*

---

## Sustentación por equipos

20 minutos. Cada equipo expone tres minutos ante la clase. Ni uno más.

Un diagnóstico que no se sabe defender no sirve para decidir. En la semana 8 se sustenta el Proyecto Integrador 1 ante la misma exigencia; esta sesión es el ensayo.

**Estructura obligatoria de los 3 minutos**

| Tiempo | Qué se dice |
|---|---|
| 0:00 – 0:30 | Qué variación se observó y en qué periodo, con la cifra. |
| 0:30 – 1:30 | Cómo se descompuso y cuál resultó el driver dominante. |
| 1:30 – 2:30 | Qué muestran las cohortes y la segmentación sobre ese driver. |
| 2:30 – 3:00 | La decisión recomendada y la limitación del análisis. |

**Reglas**

- Se presenta desde el notebook, no desde diapositivas.
- Toda afirmación va acompañada de la cifra que la respalda. Sin cifra, no vale.
- Prohibido decir «las ventas cayeron por la competencia» o cualquier explicación que los datos no permitan verificar.
- La limitación del análisis es obligatoria y se evalúa.

**Autoevaluación del equipo**

Marquen honestamente antes de exponer:

| Criterio | Sí | No |
|---|---|---|
| Nuestra descomposición reconcilia con la variación observada. | X |   |
| Toda afirmación nuestra tiene una cifra detrás. | X |   |
| Comparamos cohortes de igual antigüedad. | X |   |
| Distinguimos lo que el análisis muestra de lo que no demuestra. | X |   |
| Podemos explicar cada línea del código que ejecutamos. | X |   |

---

## Conclusiones

Escriba tres conclusiones, técnicas y de negocio, de este laboratorio.

1. **Técnica:** la descomposición precio-volumen-mezcla solo aplica de forma directa al catálogo que se repite entre los dos periodos; los productos nuevos y los discontinuados hay que cuantificarlos aparte para que el puente cuadre exactamente con la variación total observada (aquí, £236,086.10 sin diferencia residual).

2. **Técnica:** en una matriz de cohortes hay que distinguir bien entre una celda vacía (porque todavía no se alcanza esa antigüedad) y una retención realmente en cero; las cohortes solo se pueden comparar en igualdad de antigüedad, nunca en el mismo mes del calendario.

3. **De negocio:** el 2.51 % de crecimiento en ingresos que muestra el caso es una métrica de vanidad, porque esconde una contracción real de la actividad (unidades -6.72 %, facturas -3.98 %, catálogo -7.49 %) y una retención de clientes que va empeorando (de 37.64 % a 8.33 % al mes 6, entre las primeras y las últimas cohortes). La decisión comercial debería apoyarse en el driver dominante identificado, con su cifra de respaldo (productos nuevos, £2,046,903.92), y no solo en el titular de ingresos, reconociendo además que este análisis no prueba por qué ocurrieron esos cambios.
